<a href="https://colab.research.google.com/github/taraponglab/AMP-ML/blob/main/Modification_Process_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Inactive peptide → mutate to AMP-like peptide → calculate descriptors using your original method → predict KP/EC/PS using Random Forest → rank best candidates ✅

# AMP Candidate Optimization Pipeline Version History

## Version 1 (v1)

Initial mutation strategy.

Rules:

* D, E → K (increase positive charge)
* S, T, N, Q → R (increase positive charge)
* P → A (improve helix formation)
* Add terminal cysteines (C...C)
* Predict activity using KP, EC, and PS Random Forest models

Output:

* inactive_to_modified_RF_prediction_v1.csv

---

## Version 2 (v2)

Systematic mutation strategy.

Additional rules:

* Generate multiple mutation variants per inactive peptide
* Explore K-rich and R-rich variants
* Explore hydrophobic substitutions
* Remove duplicate modified sequences
* Rank candidates by Mean_RF score

Output:

* inactive_to_modified_RF_prediction_v2.csv

---

## Version 3 (v3)

Candidate prioritization.

Additional filters:

* Length ≤ 22 aa
* Charge between +2 and +9
* pI > 9
* Helix score threshold
* Proline ≤ 1
* Terminal cysteine stabilization

Output:

* inactive_to_modified_RF_prediction_v3.csv

---

## Model Information

Species-specific Random Forest models:

1. Klebsiella pneumoniae RF model
2. Escherichia coli RF model
3. Pseudomonas aeruginosa RF model

Prediction score:

Mean_RF =
(KP_RF + EC_RF + PS_RF) / 3

Higher Mean_RF indicates higher predicted AMP probability.

---

## Descriptor Calculation

Descriptors were recalculated directly from peptide sequences using Biopython and custom functions.

Features:

* Length
* Charge
* Hydrophobicity
* Molecular Weight
* Number of Cysteines
* Number of Disulfide Bridges
* Isoelectric Point
* Helix
* Turn
* Sheet
* Flexibility
* Amino Acid Composition (20 amino acids)

The original descriptors from the inactive dataset were not reused. All descriptors were recalculated from the modified sequences to ensure consistency and reproducibility.

---

## Output Naming Convention

inactive_to_modified_RF_prediction_v1.csv
inactive_to_modified_RF_prediction_v2.csv
inactive_to_modified_RF_prediction_v3.csv

or

inactive_to_modified_RF_prediction_v3_20260603_2245.csv

where:

* v3 = pipeline version
* 20260603 = date (YYYYMMDD)
* 2245 = time (HHMM)

This allows exact tracking of candidate generation runs.


In [ ]:
# =========================
# 1. SETUP
# =========================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install biopython joblib -q

import pandas as pd
import numpy as np
import joblib
from Bio.SeqUtils.ProtParam import ProteinAnalysis


# =========================
# 2. LOAD INACTIVE DATA
# =========================
inactive_df = pd.read_csv(
    "/content/drive/MyDrive/KP/KP imbalance/Inactive_373_Features.csv"
)

inactive_df.columns = inactive_df.columns.str.strip()

print(inactive_df.head())
print(inactive_df.columns)

seq_col = "Sequence"


# =========================
# 3. LOAD RF MODELS
# =========================
rf_kp = joblib.load("/content/drive/MyDrive/AMP-ML/models/klebsiella/RandomForest.pkl")
rf_ec = joblib.load("/content/drive/MyDrive/AMP-ML/models/ecoli/random_forest.pkl")
rf_ps = joblib.load("/content/drive/MyDrive/AMP-ML/models/pseudomonas/random_forest.pkl")

models = {
    "KP_RF": rf_kp,
    "EC_RF": rf_ec,
    "PS_RF": rf_ps
}

print("KP_RF expected feature names:")
print(list(rf_kp.feature_names_in_))


# =========================
# 4. FEATURE CALCULATION
# =========================
def clean_sequence(seq):
    return str(seq).strip().upper()


def calculate_features(seq):
    seq = clean_sequence(seq)
    analysis = ProteinAnalysis(seq)
    length = len(seq)

    positive = seq.count("K") + seq.count("R") + seq.count("H")
    negative = seq.count("D") + seq.count("E")

    features = {
        "Length": length,
        "Charge": positive - negative,
        "Hydrophobicity": analysis.gravy(),
        "Molecular_Weight": analysis.molecular_weight(),
        "Number_of_Cysteines": seq.count("C"),
        "Number_of_Disulfide_Bridges": seq.count("C") // 2,
        "Isoelectric_Point": analysis.isoelectric_point(),
        "Helix": analysis.secondary_structure_fraction()[0],
        "Turn": analysis.secondary_structure_fraction()[1],
        "Sheet": analysis.secondary_structure_fraction()[2],
        "Flexibility": np.mean(analysis.flexibility()) if length > 9 else 0
    }

    for aa in "ACDEFGHIKLMNPQRSTVWY":
        features[aa] = seq.count(aa) / length

    return features


# =========================
# =========================
# 5. SYSTEMATIC MUTATION LOGIC
# =========================

def generate_mutation_variants(seq, max_variants=20):
    seq = clean_sequence(seq)
    core = seq[:20]

    variants = set()

    # Strategy 1: basic mutation
    mutated = list(core)
    for i, aa in enumerate(mutated):
        if aa in ["D", "E"]:
            mutated[i] = "K"
        elif aa in ["S", "T", "N", "Q"]:
            mutated[i] = "R"
        elif aa == "P":
            mutated[i] = "A"

    base_variant = "".join(mutated)
    variants.add(base_variant)

    # Strategy 2: increase K
    for i, aa in enumerate(core):
        temp = list(base_variant)
        if aa not in ["K", "R", "C"]:
            temp[i] = "K"
            variants.add("".join(temp))

    # Strategy 3: increase R
    for i, aa in enumerate(core):
        temp = list(base_variant)
        if aa not in ["K", "R", "C"]:
            temp[i] = "R"
            variants.add("".join(temp))

    # Strategy 4: reduce proline / glycine for helix
    for i, aa in enumerate(core):
        temp = list(base_variant)
        if aa in ["P", "G"]:
            temp[i] = "A"
            variants.add("".join(temp))

    # Strategy 5: add hydrophobic balance
    for i, aa in enumerate(core):
        temp = list(base_variant)
        if aa in ["S", "T", "N", "Q", "D", "E"]:
            temp[i] = "W"
            variants.add("".join(temp))

    # Add Cys caps
    final_variants = set()
    for v in variants:
        if not v.startswith("C"):
            v = "C" + v
        if not v.endswith("C"):
            v = v + "C"

        final_variants.add(v[:22])

    return list(final_variants)[:max_variants]


# =========================
# 6. PREDICT USING EXACT MODEL FEATURES
# =========================
def predict_one_model(features, model):
    required_features = list(model.feature_names_in_)

    X = pd.DataFrame([features])

    for col in required_features:
        if col not in X.columns:
            X[col] = 0

    X = X[required_features]

    return model.predict_proba(X)[0][1]


def predict_all_models(features):
    kp = predict_one_model(features, rf_kp)
    ec = predict_one_model(features, rf_ec)
    ps = predict_one_model(features, rf_ps)

    return {
        "KP_RF": kp,
        "EC_RF": ec,
        "PS_RF": ps,
        "Mean_RF": np.mean([kp, ec, ps])
    }


# =========================
# 7. AMP CRITERIA CHECK
# =========================
def check_amp_criteria(features):
    proline_count = features["P"] * features["Length"]

    return (
        features["Length"] <= 22 and
        2 <= features["Charge"] <= 9 and
        features["Isoelectric_Point"] > 9 and
        features["Helix"] >= 0.3 and
        proline_count <= 1 and
        features["Number_of_Cysteines"] >= 2
    )


# =========================
# =========================
# 8. FULL SYSTEMATIC DESIGN PIPELINE
# =========================

results = []

for idx, row in inactive_df.iterrows():

    original_seq = clean_sequence(row[seq_col])
    original_features = calculate_features(original_seq)
    original_pred = predict_all_models(original_features)

    modified_variants = generate_mutation_variants(original_seq, max_variants=30)

    for variant_id, modified_seq in enumerate(modified_variants, start=1):

        modified_features = calculate_features(modified_seq)
        modified_pred = predict_all_models(modified_features)

        results.append({
            "Original_ID": idx + 1,
            "Variant_ID": variant_id,

            "Original_Sequence": original_seq,
            "Modified_Sequence": modified_seq,

            "Original_Length": original_features["Length"],
            "Modified_Length": modified_features["Length"],

            "Original_Charge": original_features["Charge"],
            "Modified_Charge": modified_features["Charge"],

            "Original_Hydrophobicity": round(original_features["Hydrophobicity"], 3),
            "Modified_Hydrophobicity": round(modified_features["Hydrophobicity"], 3),

            "Original_pI": round(original_features["Isoelectric_Point"], 2),
            "Modified_pI": round(modified_features["Isoelectric_Point"], 2),

            "Original_Helix": round(original_features["Helix"], 3),
            "Modified_Helix": round(modified_features["Helix"], 3),

            "Original_Cysteines": original_features["Number_of_Cysteines"],
            "Modified_Cysteines": modified_features["Number_of_Cysteines"],

            "Original_KP_RF": round(original_pred["KP_RF"], 3),
            "Original_EC_RF": round(original_pred["EC_RF"], 3),
            "Original_PS_RF": round(original_pred["PS_RF"], 3),
            "Original_Mean_RF": round(original_pred["Mean_RF"], 3),

            "Modified_KP_RF": round(modified_pred["KP_RF"], 3),
            "Modified_EC_RF": round(modified_pred["EC_RF"], 3),
            "Modified_PS_RF": round(modified_pred["PS_RF"], 3),
            "Modified_Mean_RF": round(modified_pred["Mean_RF"], 3),

            "Prediction_Increase": round(
                modified_pred["Mean_RF"] - original_pred["Mean_RF"], 3
            ),

            "Pass_AMP_Criteria": check_amp_criteria(modified_features)
        })


comparison_df = pd.DataFrame(results)

# remove duplicated modified candidates
comparison_df = comparison_df.drop_duplicates(subset=["Modified_Sequence"])

# rank best candidates
comparison_df = comparison_df.sort_values(
    by=["Pass_AMP_Criteria", "Modified_Mean_RF", "Prediction_Increase"],
    ascending=[False, False, False]
)

comparison_df.head(30)

Mounted at /content/drive
                          Sequence  Activity  Length  Charge  Hydrophobicity  \
0                EQEELENYIEHVLLHRP         0      17      -2       -1.070588   
1  APRSLRRSSCFGGRMDRIGAQSGLGCNSFRY         0      31       5       -0.587097   
2                   SSSSGLGCKVLRRH         0      14       4       -0.414286   
3        GKYGFYTHVFRLKKWIQKVIDRLGS         0      25       6       -0.416000   
4                      QAGANTRPCPS         0      11       1       -0.954545   

   Molecular_Weight  Number_of_Cysteines  Number_of_Disulfide_Bridges  \
0            2436.6                    0                            0   
1            3947.7                    2                            1   
2            1721.1                    1                            0   
3            3473.2                    0                            0   
4            1281.4                    1                            0   

   Isoelectric_Point  Helix  ...         M         N  

,Original_ID,Variant_ID,Original_Sequence,Modified_Sequence,Original_Length,Modified_Length,Original_Charge,Modified_Charge,Original_Hydrophobicity,Modified_Hydrophobicity,...,Original_KP_RF,Original_EC_RF,Original_PS_RF,Original_Mean_RF,Modified_KP_RF,Modified_EC_RF,Modified_PS_RF,Modified_Mean_RF,Prediction_Increase,Pass_AMP_Criteria
5199,192,16,QLQRIGESEDCAGIVSFLCSPDASYVNGENIAVAGYSTRL,CRLRRIGKRKKCAAIVRFLCRC,40,22,-3,9,0.037,-0.086,...,0.080,0.07,0.22,0.123,0.575,0.65,0.72,0.648,0.525,True
5208,192,25,QLQRIGESEDCAGIVSFLCSPDASYVNGENIAVAGYSTRL,CRLRRIAKRKKCAGIVRFLCRC,40,22,-3,9,0.037,-0.086,...,0.080,0.07,0.22,0.123,0.575,0.65,0.72,0.648,0.525,True
5796,213,1,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLKLRKVRVAACRFCRRC,47,22,0,8,-0.068,0.145,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5798,213,3,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVKLALRKVRVAACRFCRRC,47,22,0,8,-0.068,0.245,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5801,213,6,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLALRKRRVAACRFCRRC,47,22,0,8,-0.068,0.009,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5804,213,9,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLAKRKVRVAACRFCRRC,47,22,0,8,-0.068,0.055,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5808,213,13,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLALRKVRVAACRRCRRC,47,22,0,8,-0.068,0.073,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5809,213,14,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGKALRKVRVAACRFCRRC,47,22,0,8,-0.068,0.055,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5810,213,15,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRKVGLALRKVRVAACRFCRRC,47,22,0,8,-0.068,0.145,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5812,213,17,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLALRKVRRAACRFCRRC,47,22,0,8,-0.068,0.009,...,0.080,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True


In [ ]:
import os
import re
from datetime import datetime

save_dir = "/content/drive/MyDrive"

existing_files = os.listdir(save_dir)

versions = []

for file in existing_files:
    match = re.match(
        r"inactive_to_modified_RF_prediction_v(\d+)",
        file
    )

    if match:
        versions.append(int(match.group(1)))

next_version = max(versions) + 1 if versions else 1

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

output_file = (
    f"{save_dir}/inactive_to_modified_RF_prediction_v{next_version}_{timestamp}.csv"
)

comparison_df.to_csv(output_file, index=False)

print("="*60)
print(f"Version : v{next_version}")
print(f"Time    : {timestamp}")
print(f"Saved   : {output_file}")
print("="*60)

Version : v3
Time    : 20260603_1641
Saved   : /content/drive/MyDrive/inactive_to_modified_RF_prediction_v3_20260603_1641.csv


9,000+ systematic variants
↓
Remove duplicates
↓
Filter candidates that pass AMP criteria
↓
Rank by Modified_Mean_RF
↓
Select Top 10 only

In [ ]:
# =========================
# SELECT TOP 10 BEST CANDIDATES
# =========================

top10_df = comparison_df.copy()

# Keep only unique modified sequences
top10_df = top10_df.drop_duplicates(subset=["Modified_Sequence"])

# Optional: keep only candidates that pass AMP criteria
top10_df = top10_df[top10_df["Pass_AMP_Criteria"] == True]

# Rank by best prediction
top10_df = top10_df.sort_values(
    by=["Modified_Mean_RF", "Prediction_Increase"],
    ascending=[False, False]
)

# Select top 10
top10_df = top10_df.head(10)

top10_df

,Original_ID,Variant_ID,Original_Sequence,Modified_Sequence,Original_Length,Modified_Length,Original_Charge,Modified_Charge,Original_Hydrophobicity,Modified_Hydrophobicity,...,Original_KP_RF,Original_EC_RF,Original_PS_RF,Original_Mean_RF,Modified_KP_RF,Modified_EC_RF,Modified_PS_RF,Modified_Mean_RF,Prediction_Increase,Pass_AMP_Criteria
5199,192,16,QLQRIGESEDCAGIVSFLCSPDASYVNGENIAVAGYSTRL,CRLRRIGKRKKCAAIVRFLCRC,40,22,-3,9,0.037,-0.086,...,0.08,0.07,0.22,0.123,0.575,0.65,0.72,0.648,0.525,True
5208,192,25,QLQRIGESEDCAGIVSFLCSPDASYVNGENIAVAGYSTRL,CRLRRIAKRKKCAGIVRFLCRC,40,22,-3,9,0.037,-0.086,...,0.08,0.07,0.22,0.123,0.575,0.65,0.72,0.648,0.525,True
5796,213,1,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLKLRKVRVAACRFCRRC,47,22,0,8,-0.068,0.145,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5798,213,3,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVKLALRKVRVAACRFCRRC,47,22,0,8,-0.068,0.245,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5801,213,6,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLALRKRRVAACRFCRRC,47,22,0,8,-0.068,0.009,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5804,213,9,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLAKRKVRVAACRFCRRC,47,22,0,8,-0.068,0.055,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5808,213,13,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLALRKVRVAACRRCRRC,47,22,0,8,-0.068,0.073,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5809,213,14,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGKALRKVRVAACRFCRRC,47,22,0,8,-0.068,0.055,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5810,213,15,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRKVGLALRKVRVAACRFCRRC,47,22,0,8,-0.068,0.145,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True
5812,213,17,TAVGLPLQDVSVAACNFCRRPVHFELMSEWERSYFGNMGPQYVTTYA,CRAVGLALRKVRRAACRFCRRC,47,22,0,8,-0.068,0.009,...,0.08,0.07,0.22,0.123,0.575,0.64,0.71,0.642,0.518,True


Step 1: Input Inactive Peptides

Inactive peptide sequences are imported from the inactive peptide dataset. Only the peptide sequence is used as input. All descriptors are recalculated after modification to ensure consistency.

Step 2: Descriptor Calculation

For each peptide sequence, physicochemical descriptors are calculated, including:

Length
Net Charge
Hydrophobicity
Molecular Weight
Number of Cysteines
Number of Disulfide Bridges
Isoelectric Point (pI)
Helix Content
Turn Content
Sheet Content
Flexibility
Amino Acid Composition (20 amino acids)

These descriptors are generated using the same feature set employed during Random Forest model training.

Step 3: Systematic Mutation

Multiple mutation strategies are applied to generate peptide variants:

D/E → K to increase positive charge
S/T/N/Q → R to increase cationicity
P/G → A to improve helix formation
Additional K substitutions
Additional R substitutions
Hydrophobic optimization using W or F substitutions
Addition of terminal cysteine residues (C...C) for potential stabilization

This produces multiple candidate peptides from each parent sequence.

Step 4: Activity Prediction

Each modified peptide is evaluated using three independently trained Random Forest models:

Klebsiella pneumoniae model
Escherichia coli model
Pseudomonas aeruginosa model

A combined score is calculated as:

Mean_RF = (KP_RF + EC_RF + PS_RF) / 3

where higher values indicate greater predicted AMP probability.

Step 5: AMP Filtering Criteria

Candidates are filtered according to predefined AMP-like characteristics:

Length ≤ 22 amino acids
Charge between +2 and +9
pI > 9
Helix content ≥ 0.30
Proline count ≤ 1
At least 2 cysteine residues

Only candidates satisfying all criteria are retained.

Evolutionary Optimization Strategy
Round 1

Inactive peptides are mutated to generate multiple variants.

Selection:
Top 100 candidates ranked by:

Mean_RF
Prediction Increase
Round 2

The Top 100 candidates are used as parent sequences and re-optimized.

Selection:
Top 20 candidates.

Round 3

The Top 20 candidates undergo a final optimization round.

Selection:
Final Top 10 candidates.

This evolutionary approach progressively enriches peptide sequences with favorable AMP-like properties while maintaining diversity.

Candidate Ranking

Candidates are ranked using:

AMP Criteria Pass Status
Mean_RF Probability
Prediction Improvement Relative to Parent Sequence

The final Top 10 peptides represent the highest-scoring candidates generated by the optimization pipeline.

Reproducibility and Version Control

Each run is automatically saved using a versioned filename:

final_top10_evolutionary_RF_candidates_v1.csv

final_top10_evolutionary_RF_candidates_v2.csv

final_top10_evolutionary_RF_candidates_v3.csv

Optionally, timestamps are appended:

final_top10_evolutionary_RF_candidates_v3_20260603_2245.csv

This ensures that all optimization runs are traceable and reproducible.



In [1]:
# =========================
# 1. SETUP
# =========================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install biopython joblib -q

import pandas as pd
import numpy as np
import joblib
import os
import re
from datetime import datetime
from Bio.SeqUtils.ProtParam import ProteinAnalysis


# =========================
# 2. LOAD INACTIVE DATA
# =========================
inactive_df = pd.read_csv(
    "/content/drive/MyDrive/KP/KP imbalance/Inactive_373_Features.csv"
)

inactive_df.columns = inactive_df.columns.str.strip()

seq_col = "Sequence"

print(inactive_df.head())
print(inactive_df.columns)


# =========================
# 3. LOAD RANDOM FOREST MODELS
# =========================
rf_kp = joblib.load("/content/drive/MyDrive/AMP-ML/models/klebsiella/RandomForest.pkl")
rf_ec = joblib.load("/content/drive/MyDrive/AMP-ML/models/ecoli/random_forest.pkl")
rf_ps = joblib.load("/content/drive/MyDrive/AMP-ML/models/pseudomonas/random_forest.pkl")


# =========================
# 4. BASIC FUNCTIONS
# =========================
def clean_sequence(seq):
    return str(seq).strip().upper()


def calculate_features(seq):
    seq = clean_sequence(seq)
    analysis = ProteinAnalysis(seq)
    length = len(seq)

    features = {
        "Length": length,
        "Charge": seq.count("K") + seq.count("R") + seq.count("H") - seq.count("D") - seq.count("E"),
        "Hydrophobicity": analysis.gravy(),
        "Molecular_Weight": analysis.molecular_weight(),
        "Number_of_Cysteines": seq.count("C"),
        "Number_of_Disulfide_Bridges": seq.count("C") // 2,
        "Isoelectric_Point": analysis.isoelectric_point(),
        "Helix": analysis.secondary_structure_fraction()[0],
        "Turn": analysis.secondary_structure_fraction()[1],
        "Sheet": analysis.secondary_structure_fraction()[2],
        "Flexibility": np.mean(analysis.flexibility()) if length > 9 else 0
    }

    for aa in "ACDEFGHIKLMNPQRSTVWY":
        features[aa] = seq.count(aa) / length

    return features


def predict_one_model(features, model):
    required_features = list(model.feature_names_in_)

    X = pd.DataFrame([features])

    for col in required_features:
        if col not in X.columns:
            X[col] = 0

    X = X[required_features]

    return model.predict_proba(X)[0][1]


def predict_all_models(features):
    kp = predict_one_model(features, rf_kp)
    ec = predict_one_model(features, rf_ec)
    ps = predict_one_model(features, rf_ps)

    return {
        "KP_RF": kp,
        "EC_RF": ec,
        "PS_RF": ps,
        "Mean_RF": np.mean([kp, ec, ps])
    }


# =========================
# 5. AMP CRITERIA
# =========================
def check_amp_criteria(features):
    proline_count = features["P"] * features["Length"]

    return (
        features["Length"] <= 22 and
        2 <= features["Charge"] <= 9 and
        features["Isoelectric_Point"] > 9 and
        features["Helix"] >= 0.30 and
        proline_count <= 1 and
        features["Number_of_Cysteines"] >= 2
    )


# =========================
# =========================
# 6. DESCRIPTOR-GUIDED MUTATION VARIANTS
# =========================

def add_cys_guided(seq):
    """
    Add cysteine only when needed:
    - 0 Cys → add C at both ends
    - 1 Cys → add one terminal C
    - ≥2 Cys → keep as is
    """
    seq = clean_sequence(seq)
    cys_count = seq.count("C")

    if cys_count == 0:
        seq = "C" + seq + "C"
    elif cys_count == 1:
        if not seq.startswith("C"):
            seq = "C" + seq
        elif not seq.endswith("C"):
            seq = seq + "C"

    return seq[:22]


def generate_mutation_variants(seq, max_variants=50):
    seq = clean_sequence(seq)
    core = seq[:20]

    variants = set()

    original_features = calculate_features(core)

    # Base sequence
    variants.add(core)

    # Step 1: improve charge and pI only when needed
    if original_features["Charge"] < 4 or original_features["Isoelectric_Point"] < 9:
        for i, aa in enumerate(core):
            temp = list(core)

            if aa in ["D", "E"]:
                temp[i] = "K"
                variants.add("".join(temp))

            elif aa in ["S", "T", "N", "Q"]:
                temp[i] = "R"
                variants.add("".join(temp))

    # Step 2: improve helix only when needed
    if original_features["Helix"] < 0.30:
        for i, aa in enumerate(core):
            temp = list(core)

            if aa in ["P", "G"]:
                temp[i] = "A"
                variants.add("".join(temp))

    # Step 3: add limited hydrophobic/aromatic residues only when hydrophobicity is too low
    if original_features["Hydrophobicity"] < -0.5:
        for i, aa in enumerate(core):
            temp = list(core)

            if aa in ["S", "T", "N", "Q", "D", "E"]:
                temp[i] = "W"
                variants.add("".join(temp))

            temp = list(core)

            if aa in ["S", "T", "N", "Q", "D", "E"]:
                temp[i] = "F"
                variants.add("".join(temp))

    # Step 4: generate combined rational variant
    combined = list(core)

    for i, aa in enumerate(combined):
        if aa in ["D", "E"]:
            combined[i] = "K"
        elif aa in ["P", "G"]:
            combined[i] = "A"
        elif aa in ["S", "T", "N", "Q"] and original_features["Charge"] < 6:
            combined[i] = "R"

    variants.add("".join(combined))

    # Add guided cysteine only
    final_variants = set()

    for v in variants:
        v = add_cys_guided(v)
        f = calculate_features(v)

        # avoid over-cysteine candidates
        if f["Number_of_Cysteines"] <= 2:
            final_variants.add(v)

    return list(final_variants)[:max_variants]


# =========================
# 7. EVOLUTIONARY OPTIMIZATION
# =========================
def optimize_round(input_df, round_name, top_n=100):
    results = []

    for idx, row in input_df.iterrows():

        if "Modified_Sequence" in input_df.columns:
            original_seq = clean_sequence(row["Modified_Sequence"])
        else:
            original_seq = clean_sequence(row[seq_col])

        original_features = calculate_features(original_seq)
        original_pred = predict_all_models(original_features)

        variants = generate_mutation_variants(original_seq, max_variants=50)

        for variant_id, modified_seq in enumerate(variants, start=1):

            modified_features = calculate_features(modified_seq)
            modified_pred = predict_all_models(modified_features)

            results.append({
                "Round": round_name,
                "Parent_ID": idx + 1,
                "Variant_ID": variant_id,

                "Original_Sequence": original_seq,
                "Modified_Sequence": modified_seq,

                "Modified_Length": modified_features["Length"],
                "Modified_Charge": modified_features["Charge"],
                "Modified_Hydrophobicity": round(modified_features["Hydrophobicity"], 3),
                "Modified_pI": round(modified_features["Isoelectric_Point"], 2),
                "Modified_Helix": round(modified_features["Helix"], 3),
                "Modified_Cysteines": modified_features["Number_of_Cysteines"],

                "Original_Mean_RF": round(original_pred["Mean_RF"], 3),

                "Modified_KP_RF": round(modified_pred["KP_RF"], 3),
                "Modified_EC_RF": round(modified_pred["EC_RF"], 3),
                "Modified_PS_RF": round(modified_pred["PS_RF"], 3),
                "Modified_Mean_RF": round(modified_pred["Mean_RF"], 3),

                "Prediction_Increase": round(
                    modified_pred["Mean_RF"] - original_pred["Mean_RF"], 3
                ),

                "Pass_AMP_Criteria": check_amp_criteria(modified_features)
            })

    df = pd.DataFrame(results)

    df = df.drop_duplicates(subset=["Modified_Sequence"])
    df = df[df["Pass_AMP_Criteria"] == True]

    df = df.sort_values(
        by=["Modified_Mean_RF", "Prediction_Increase"],
        ascending=[False, False]
    )

    return df.head(top_n)


# =========================
# =========================
# 8. RUN 3 ROUNDS
# =========================

round1_df = optimize_round(
    inactive_df,
    round_name="Round_1_from_inactive",
    top_n=100
)

print("Round 1 completed:", round1_df.shape)
display(round1_df.head(10))


round2_df = optimize_round(
    round1_df,
    round_name="Round_2_from_top100",
    top_n=20
)

print("Round 2 completed:", round2_df.shape)
display(round2_df.head(10))


round3_df = optimize_round(
    round2_df,
    round_name="Round_3_from_top20",
    top_n=10
)

print("Round 3 completed:", round3_df.shape)
display(round3_df)


# =========================
# 9. SAVE ALL ROUNDS + DOWNLOAD
# =========================
from google.colab import files
import zipfile

save_dir = "/content/drive/MyDrive/AMP_Evolutionary_RF_Results"
os.makedirs(save_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

round1_file = f"{save_dir}/round1_top100_RF_candidates_{timestamp}.csv"
round2_file = f"{save_dir}/round2_top20_RF_candidates_{timestamp}.csv"
round3_file = f"{save_dir}/round3_final_top10_RF_candidates_{timestamp}.csv"
combined_file = f"{save_dir}/all_rounds_RF_candidates_{timestamp}.csv"

# Save each round
round1_df.to_csv(round1_file, index=False)
round2_df.to_csv(round2_file, index=False)
round3_df.to_csv(round3_file, index=False)

# Save combined file
combined_df = pd.concat(
    [round1_df, round2_df, round3_df],
    ignore_index=True
)

combined_df.to_csv(combined_file, index=False)

# Create ZIP file
zip_file = f"{save_dir}/AMP_Evolutionary_RF_All_Rounds_{timestamp}.zip"

with zipfile.ZipFile(zip_file, "w") as zipf:
    zipf.write(round1_file, os.path.basename(round1_file))
    zipf.write(round2_file, os.path.basename(round2_file))
    zipf.write(round3_file, os.path.basename(round3_file))
    zipf.write(combined_file, os.path.basename(combined_file))

print("="*70)
print("Saved all evolutionary RF candidate files")
print(f"Round 1 file : {round1_file}")
print(f"Round 2 file : {round2_file}")
print(f"Round 3 file : {round3_file}")
print(f"Combined file: {combined_file}")
print(f"ZIP file     : {zip_file}")
print("="*70)

# Download ZIP directly
files.download(zip_file)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.2 MB/s eta 0:00:00
                          Sequence  Activity  Length  Charge  Hydrophobicity  \
0                EQEELENYIEHVLLHRP         0      17      -2       -1.070588   
1  APRSLRRSSCFGGRMDRIGAQSGLGCNSFRY         0      31       5       -0.587097   
2                   SSSSGLGCKVLRRH         0      14       4       -0.414286   
3        GKYGFYTHVFRLKKWIQKVIDRLGS         0      25       6       -0.416000   
4                      QAGANTRPCPS         0      11       1       -0.954545   

   Molecular_Weight  Number_of_Cysteines  Number_of_Disulfide_Bridges  \
0            2436.6                    0                            0   
1            3947.7                    2                            1   
2            1721.1                    1                            0   
3            3473.2                    0                            0   
4            1281.4                    1           

,Round,Parent_ID,Variant_ID,Original_Sequence,Modified_Sequence,Modified_Length,Modified_Charge,Modified_Hydrophobicity,Modified_pI,Modified_Helix,Modified_Cysteines,Original_Mean_RF,Modified_KP_RF,Modified_EC_RF,Modified_PS_RF,Modified_Mean_RF,Prediction_Increase,Pass_AMP_Criteria
1094,Round_1_from_inactive,78,15,KSCDMKALREYCSVVRN,KRCKMKALRKYCRVVRR,17,9,-1.088,11.48,0.412,2,0.137,0.540,0.64,0.73,0.637,0.500,True
2131,Round_1_from_inactive,157,6,KPSPAASKEYFQKVNQ,CKARAAARKKYFRKVRRC,18,9,-1.122,11.48,0.444,2,0.200,0.540,0.64,0.73,0.637,0.437,True
1103,Round_1_from_inactive,79,7,DLELQKIAEKFSQRG,CKLKLRKIAKKFRRRAC,17,9,-0.824,11.59,0.529,2,0.100,0.535,0.64,0.73,0.635,0.535,True
2620,Round_1_from_inactive,193,17,CVPSREPKDMTTFRSA,CVARRKAKKMRRFRRAC,17,9,-1.141,12.00,0.412,2,0.112,0.535,0.64,0.73,0.635,0.523,True
564,Round_1_from_inactive,46,15,AMDLELQKIAEKFSQRG,CAMKLKLRKIAKKFRRRAC,19,9,-0.542,11.59,0.579,2,0.105,0.540,0.64,0.72,0.633,0.528,True
2587,Round_1_from_inactive,191,5,DSRTAKSGGLSVEVCGPALSQQWKFTLNLQQ,CKRRRAKRAALRVKVCAAALR,21,9,-0.329,12.00,0.524,2,0.117,0.540,0.64,0.72,0.633,0.517,True
427,Round_1_from_inactive,32,19,GGNPQQPQAPPAGQPQGPPRPPQGGRPSRPPQ,CAARARRARAAAAARARAAARC,22,7,-0.141,12.00,0.591,2,0.347,0.540,0.63,0.73,0.633,0.287,True
1009,Round_1_from_inactive,75,18,QEANQTLVGIDWQHL,CRKARRRLVAIKWRHLC,17,8,-0.559,11.84,0.353,2,0.100,0.540,0.63,0.72,0.630,0.530,True
118,Round_1_from_inactive,9,6,KGVPTSTVYAQILFEENQL,CKAVARRRVYARILFKKRRLC,21,9,-0.300,11.72,0.381,2,0.105,0.540,0.64,0.71,0.630,0.525,True
1741,Round_1_from_inactive,126,4,AKGAPTSTVYAQILFEENKL,CAKAAARRRVYARILFKKRKLC,22,9,-0.286,11.48,0.500,2,0.107,0.540,0.64,0.71,0.630,0.523,True


Round 2 completed: (20, 18)


,Round,Parent_ID,Variant_ID,Original_Sequence,Modified_Sequence,Modified_Length,Modified_Charge,Modified_Hydrophobicity,Modified_pI,Modified_Helix,Modified_Cysteines,Original_Mean_RF,Modified_KP_RF,Modified_EC_RF,Modified_PS_RF,Modified_Mean_RF,Prediction_Increase,Pass_AMP_Criteria
0,Round_2_from_top100,1095,1,KRCKMKALRKYCRVVRR,KRCKMKALRKYCRVVRR,17,9,-1.088,11.48,0.412,2,0.637,0.540,0.64,0.73,0.637,0.000,True
1,Round_2_from_top100,2132,1,CKARAAARKKYFRKVRRC,CKARAAARKKYFRKVRRC,18,9,-1.122,11.48,0.444,2,0.637,0.540,0.64,0.73,0.637,0.000,True
2,Round_2_from_top100,1104,1,CKLKLRKIAKKFRRRAC,CKLKLRKIAKKFRRRAC,17,9,-0.824,11.59,0.529,2,0.635,0.535,0.64,0.73,0.635,0.000,True
3,Round_2_from_top100,2621,1,CVARRKAKKMRRFRRAC,CVARRKAKKMRRFRRAC,17,9,-1.141,12.00,0.412,2,0.635,0.535,0.64,0.73,0.635,0.000,True
15,Round_2_from_top100,526,1,CRKVVIRRACHARKRIIARAA,CRKVVIRRACHARKRIIARA,20,9,-0.195,12.00,0.300,2,0.630,0.540,0.64,0.72,0.633,0.003,True
18,Round_2_from_top100,5063,1,CKMAARLKVKALRAAARRHKAC,CKMAARLKVKALRAAARRHKC,21,9,-0.348,11.58,0.619,2,0.630,0.540,0.64,0.72,0.633,0.003,True
25,Round_2_from_top100,2978,1,CAMAKKKRKAARARKIRAIFFC,CAMAKKKRKAARARKIRAIFC,21,9,-0.381,11.59,0.571,2,0.630,0.540,0.64,0.72,0.633,0.003,True
27,Round_2_from_top100,2526,1,CARVCHAHARLRKAFRKARLA,CARVCHAHARLRKAFRKARL,20,9,-0.405,11.84,0.450,2,0.630,0.540,0.64,0.72,0.633,0.003,True
34,Round_2_from_top100,790,1,CARARARAAARRAAHRRAAAAC,CARARARAAARRAAHRRAAAC,21,8,-0.471,12.00,0.524,2,0.630,0.540,0.63,0.73,0.633,0.003,True
4,Round_2_from_top100,565,1,CAMKLKLRKIAKKFRRRAC,CAMKLKLRKIAKKFRRRAC,19,9,-0.542,11.59,0.579,2,0.633,0.540,0.64,0.72,0.633,0.000,True


Round 3 completed: (10, 18)


,Round,Parent_ID,Variant_ID,Original_Sequence,Modified_Sequence,Modified_Length,Modified_Charge,Modified_Hydrophobicity,Modified_pI,Modified_Helix,Modified_Cysteines,Original_Mean_RF,Modified_KP_RF,Modified_EC_RF,Modified_PS_RF,Modified_Mean_RF,Prediction_Increase,Pass_AMP_Criteria
0,Round_3_from_top20,1,1,KRCKMKALRKYCRVVRR,KRCKMKALRKYCRVVRR,17,9,-1.088,11.48,0.412,2,0.637,0.540,0.64,0.73,0.637,0.0,True
1,Round_3_from_top20,2,1,CKARAAARKKYFRKVRRC,CKARAAARKKYFRKVRRC,18,9,-1.122,11.48,0.444,2,0.637,0.540,0.64,0.73,0.637,0.0,True
2,Round_3_from_top20,3,1,CKLKLRKIAKKFRRRAC,CKLKLRKIAKKFRRRAC,17,9,-0.824,11.59,0.529,2,0.635,0.535,0.64,0.73,0.635,0.0,True
3,Round_3_from_top20,4,1,CVARRKAKKMRRFRRAC,CVARRKAKKMRRFRRAC,17,9,-1.141,12.00,0.412,2,0.635,0.535,0.64,0.73,0.635,0.0,True
4,Round_3_from_top20,16,1,CRKVVIRRACHARKRIIARA,CRKVVIRRACHARKRIIARA,20,9,-0.195,12.00,0.300,2,0.633,0.540,0.64,0.72,0.633,0.0,True
5,Round_3_from_top20,19,1,CKMAARLKVKALRAAARRHKC,CKMAARLKVKALRAAARRHKC,21,9,-0.348,11.58,0.619,2,0.633,0.540,0.64,0.72,0.633,0.0,True
6,Round_3_from_top20,26,1,CAMAKKKRKAARARKIRAIFC,CAMAKKKRKAARARKIRAIFC,21,9,-0.381,11.59,0.571,2,0.633,0.540,0.64,0.72,0.633,0.0,True
7,Round_3_from_top20,28,1,CARVCHAHARLRKAFRKARL,CARVCHAHARLRKAFRKARL,20,9,-0.405,11.84,0.450,2,0.633,0.540,0.64,0.72,0.633,0.0,True
8,Round_3_from_top20,35,1,CARARARAAARRAAHRRAAAC,CARARARAAARRAAHRRAAAC,21,8,-0.471,12.00,0.524,2,0.633,0.540,0.63,0.73,0.633,0.0,True
9,Round_3_from_top20,5,1,CAMKLKLRKIAKKFRRRAC,CAMKLKLRKIAKKFRRRAC,19,9,-0.542,11.59,0.579,2,0.633,0.540,0.64,0.72,0.633,0.0,True


Saved all evolutionary RF candidate files
Round 1 file : /content/drive/MyDrive/AMP_Evolutionary_RF_Results/round1_top100_RF_candidates_20260604_1100.csv
Round 2 file : /content/drive/MyDrive/AMP_Evolutionary_RF_Results/round2_top20_RF_candidates_20260604_1100.csv
Round 3 file : /content/drive/MyDrive/AMP_Evolutionary_RF_Results/round3_final_top10_RF_candidates_20260604_1100.csv
Combined file: /content/drive/MyDrive/AMP_Evolutionary_RF_Results/all_rounds_RF_candidates_20260604_1100.csv
ZIP file     : /content/drive/MyDrive/AMP_Evolutionary_RF_Results/AMP_Evolutionary_RF_All_Rounds_20260604_1100.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>